In [1]:
using Pkg
Pkg.activate(".")
using SparseArrays
using Arrhenius
using BenchmarkTools

  Activating project at `~/Dendron/notes/assets/scripts`


In [2]:
abstract type Reactions end

struct ForwardReactions{Ty<:AbstractArray,sTy<:AbstractArray}<:Reactions
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    vk::sTy
end

struct ReversibleReactions{Ty<:AbstractArray,sTy<:AbstractArray}<:Reactions
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    product_stoich_coeffs::sTy
    reactant_stoich_coeffs::sTy
    vk::sTy
end

In [3]:
gas = CreateSolution("gri30.yaml",sparseCoeffs=true);
kf = zeros(gas.n_reactions);
kr = zeros(gas.n_reactions);
C = rand(gas.n_species)
wdot = zeros(gas.n_species)
T =900

900

In [4]:
forwardReactions = ForwardReactions(gas.kinetics.Arrhenius_coeffs[.!(gas.kinetics.is_reversible),:],
                    gas.kinetics.reactant_orders[:,.!(gas.kinetics.is_reversible)],
                    gas.kinetics.vk[:,.!(gas.kinetics.is_reversible)],#sparse(transpose(dense(gas.kinetics.vk))),
                    )

reversibleReactions = ReversibleReactions(gas.kinetics.Arrhenius_coeffs[(gas.kinetics.is_reversible),:],
    gas.kinetics.reactant_orders[:,(gas.kinetics.is_reversible)],
    gas.kinetics.vk[:,(gas.kinetics.is_reversible)],#sparse(transpose(dense(gas.kinetics.vk))),
    gas.kinetics.product_stoich_coeffs[:,(gas.kinetics.is_reversible)],
    gas.kinetics.reactant_stoich_coeffs[:,(gas.kinetics.is_reversible)],
    )

ReversibleReactions{Matrix{Float64}, SparseMatrixCSC{Float64, Int64}}([1.2e11 -1.0 0.0; 5.0e11 -1.0 0.0; … ; 2.55e7 0.255 -943.0; 1.927e10 -0.32 0.0], sparse([3, 2, 3, 1, 3, 3, 7, 3, 8, 3  …  2, 50, 2, 50, 5, 50, 7, 50, 13, 50], [1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 18, 18, 19, 19, 20, 20, 21, 21, 22, 22, 23, 23, 24, 24, 25, 25, 26, 26, 27, 27, 28, 28, 29, 29, 30, 30, 31, 31, 32, 32, 33, 33, 34, 34, 35, 35, 35, 36, 36, 36, 37, 37, 37, 38, 38, 39, 40, 40, 41, 41, 42, 42, 43, 43, 44, 44, 45, 45, 46, 46, 47, 47, 48, 48, 49, 49, 50, 50, 51, 51, 52, 52, 53, 53, 54, 54, 55, 55, 56, 56, 57, 57, 58, 58, 59, 59, 60, 60, 61, 61, 62, 62, 63, 63, 64, 64, 65, 65, 66, 66, 67, 67, 68, 68, 69, 69, 70, 70, 71, 71, 72, 72, 73, 73, 74, 74, 75, 75, 76, 76, 77, 77, 78, 78, 79, 79, 80, 80, 81, 81, 82, 82, 83, 83, 84, 84, 85, 86, 87, 87, 88, 88, 89, 89, 90, 90, 91, 91, 92, 92, 93, 93, 94, 94, 95, 95, 96, 96, 97, 97, 98, 98, 99, 99,

In [10]:
for g in gas.kinetics.is_reversible
    if g
        println("at_least_1")
    end
end

In [103]:



function forwardRate!(kf::AbstractVector, arr_coeffs::AbstractArray,T::Real)
    logT = log(T)
    Eexp = (4184.0 / Arrhenius.R / T)
    for i in eachindex(kf) 
        @inbounds  kf[i] = arr_coeffs[i, 1]  * exp(arr_coeffs[i, 2] * logT - arr_coeffs[i, 3] * Eexp)
    end
    #@. kf = @view(arr_coeffs[:, 1])  * exp(@view(arr_coeffs[:, 2]) * logT - @view(arr_coeffs[:, 3]) * Eexp)
end
function kf_Pi_c_rectantOrder!(kf,C,reactant_orders,i_reactant)
    for i in eachindex(kf)
        for j in reactant_orders.colptr[i]:reactant_orders.colptr[i+1]-1
            @inbounds kf[i] *= C[reactant_orders.rowval[j]]^reactant_orders.nzval[j]
        end
    end
end
function vk_kf!(wdot,kf,vk) # TODO use likwid to compare the two methods
    # transposed vk - workflow 
    # for isp in eachindex(wdot) 
    #     for j in vk.colptr[isp]:vk.colptr[isp+1]-1
    #         wdot[isp] += vk.nzval[j] * kf[vk.rowval[j]]
    #     end
    # end 
    for ir in eachindex(kf)
        for j in vk.colptr[ir]:vk.colptr[ir+1]-1
            wdot[vk.rowval[j]] += vk.nzval[j] * kf[ir]
        end
    end
    return nothing
end
function wdot_func!(wdot::AbstractVector,kf::AbstractVector,C::AbstractVector,forwardReactions::ForwardReactions ,T::Real)
    wdot .= 0.0
    forwardRate!(kf, forwardReactions.arrhenius_coeffs,T)
    kf_Pi_c_rectantOrder!(kf,C,forwardReactions.reactant_orders,forwardReactions.i_reactant)
    vk_kf!(wdot,kf,forwardReactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$kf,$C,$forwardReactions,$T) evals = 200 seconds = 20

BenchmarkTools.Trial: 9672 samples with 200 evaluations.
 Range (min … max):   7.895 μs … 240.261 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):      8.802 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   10.233 μs ±   3.708 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▇▆▆▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▃▂▂▂▂▂▂▃▂▂▂▃▃▃▃▂▃▃▂▂▂▂▂▂▂▁▁▁▁ ▁         ▂
  ██████████████████████████████████████████████████████▆▇▇▆▁▇ █
  7.9 μs        Histogram: log(frequency) by time        18 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

# Comparison to old wdot

In [95]:
function wdot_func(reaction, T, C)

    @inbounds _kf = @. @view(reaction.Arrhenius_coeffs[:, 1]) * exp(
        @view(reaction.Arrhenius_coeffs[:, 2]) * log(T) -
        @view(reaction.Arrhenius_coeffs[:, 3]) * (4184.0 / R / T),
    )
    for i = 1:reaction.n_reactions
        @inbounds for j in reaction.i_reactant[i]
            @inbounds _kf[i] *= C[j]^reaction.reactant_orders[j, i]
        end
    end

    #if get_qdot
    #    return _kf - _kr
    #end
    return reaction.vk *  (_kf ) #omega_dot_k
end

@benchmark wdot_func($(gas.kinetics),$T,$C)

BenchmarkTools.Trial: 10000 samples with 1 evaluation.
 Range (min … max):  16.061 μs … 848.551 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     17.898 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   22.591 μs ±  17.127 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▅▄▃▂▂▃▄▅▄▃▂▂▂▂▁▁▁                                           ▁
  ████████████████████████▇███▇▇▇▇▇▇▇▆▇▆▇▆▆▆▅▅▅▆▆▅▅▄▄▄▄▅▅▄▂▃▄▃ █
  16.1 μs       Histogram: log(frequency) by time      66.7 μs <

 Memory estimate: 3.17 KiB, allocs estimate: 2.

In [94]:
wdot_func!(wdot,kf,C,forwardReactions,T)
wdot-wdot_func((gas.kinetics),T,C)

53-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [98]:
@benchmark log($T)

BenchmarkTools.Trial: 10000 samples with 996 evaluations.
 Range (min … max):  22.336 ns … 221.705 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     22.570 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   24.274 ns ±   8.192 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▄▂▁                                                         ▁
  █████▇▇▇▇▇▇▇▇▇▇▇▆▄▆▅▅▅▆▅▆▅▄▄▅▄▄▄▄▄▆▆▅▅▄▄▅▃▃▄▃▄▅▅▅▄▄▅▄▃▄▃▃▂▃▄ █
  22.3 ns       Histogram: log(frequency) by time      60.4 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [99]:
@benchmark (4184.0 / $R / $T)

BenchmarkTools.Trial: 10000 samples with 1000 evaluations.
 Range (min … max):  2.458 ns … 467.267 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.474 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   3.052 ns ±   7.109 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▃▄▂▁              ▃▃▄▄▄▄▃▃▃▂▁▁                             ▂
  ██████▇▆▆▆▇▆▇▆▅▆▆▆▇██████████████▇▇▇▇▆▆▆▅▅▄▅▄▄▄▄▄▄▄▃▃▁▁▁▁▁▄ █
  2.46 ns      Histogram: log(frequency) by time      4.37 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

# reversibleReaction

In [ ]:
function kr!(kf,s0_R,h_RT,vk,vk_sum,T)

end



function wdot_func!(wdot::AbstractVector,kf::AbstractVector,C::AbstractVector,reversibleReactions::ReversibleReactions,h_RT::AbstractVector ,s0_R::AbstractVector,T::Real)
    wdot .= 0.0
    forwardRate!(kf, reversibleReactions.arrhenius_coeffs,T)
    kf_Pi_c_rectantOrder!(kf,C,reversibleReactions.reactant_orders,reversibleReactions.i_reactant)
    kr!(kf,s0_R,h_RT,reversibleReactions.vk,reversibleReactions.vk_sum,T)
    vk_kf!(wdot,kf,reversibleReactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$kf,$C,$forwardReactions,$T) evals = 200 seconds = 20

In [15]:
vk_trans = sparse(gas.kinetics.vk')
@benchmark kf .= $(vk_trans) * $(C)

BenchmarkTools.Trial: 10000 samples with 10 evaluations.
 Range (min … max):  1.061 μs …  1.418 ms  ┊ GC (min … max):  0.00% … 99.49%
 Time  (median):     1.301 μs              ┊ GC (median):     0.00%
 Time  (mean ± σ):   2.332 μs ± 30.625 μs  ┊ GC (mean ± σ):  31.86% ±  2.43%

  ▄▇▇█▅▄▃▃▃▂▂▁▁▁▁▁▁▁                                         ▂
  █████████████████████▇▇▇▆▇▇▆▆▆▆▇▅▇▇▆▇██▇▆▆▆▆▇█▇▇▆▆▆▆▆▄▅▅▄▅ █
  1.06 μs      Histogram: log(frequency) by time     5.27 μs <

 Memory estimate: 2.70 KiB, allocs estimate: 2.